# Structural Design Patterns
### Adapter | Decorator | Facade | Proxy | Composite | Bridge

> **One coherent system:** ShopFlow -- 500k-user e-commerce platform.

*Run each cell with **Shift + Enter***

## Setup

In [ ]:
from __future__ import annotations
import time, functools
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Callable

---
## 1 · Adapter

### Mental Model -- 'The Power Socket Adapter'

```
WHAT   Convert one interface into another that the client expects.
       Makes incompatible classes work together WITHOUT changing source code.
WHY    You can't change a third-party library's interface.
HOW    Wrap the incompatible class; expose the expected interface.
WHEN   SDK migrations | legacy system integrations | third-party wrappers
```

```
Client expects:     Existing class:          Adapter bridges:
charge(amount,      BraintreeSDK             BraintreeAdapter
  currency)->str    .transaction_sale(...)   .charge(amount, currency)
                                              -> calls braintree internally
```

**Key insight:** The client knows only the target interface.
The adapter is the ONLY translation layer -- write it once, not 47 times.

### Real-World Scenario -- ShopFlow Payment SDK Migration

**Incident:** ShopFlow used Braintree in 47 payment-related call sites.
Stripe offered better rates. Migrating required changing all 47 files.

**Fix:** Introduce a `PaymentGateway` interface. Write a `BraintreeAdapter`
and then a `StripeAdapter`. All 47 callers used `PaymentGateway` -- zero changes.

In [ ]:
# BEFORE -- Braintree SDK used directly in 47 places

class BraintreeSDK:
    # Third-party SDK -- we cannot change this
    def transaction_sale(self, amount_cents: int, currency: str, nonce: str) -> dict:
        return {'id': 'bt_txn_abc', 'status': 'settled', 'amount': amount_cents / 100}

def checkout_BAD(cart_total: float, nonce: str) -> str:
    sdk = BraintreeSDK()
    result = sdk.transaction_sale(
        amount_cents=int(cart_total * 100),
        currency='USD',
        nonce=nonce,
    )
    if result['status'] == 'settled': return result['id']
    raise RuntimeError('Payment failed')
# 47 places calling transaction_sale directly -- migration = 47 edits.

In [ ]:
# AFTER -- Adapter Pattern

class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, amount: float, currency: str, token: str) -> str:
        # Returns a transaction ID
        ...

class BraintreeAdapter(PaymentGateway):
    def __init__(self) -> None: self._sdk = BraintreeSDK()
    def charge(self, amount: float, currency: str, token: str) -> str:
        result = self._sdk.transaction_sale(
            amount_cents=int(amount * 100), currency=currency, nonce=token)
        if result['status'] != 'settled': raise RuntimeError(f'Payment failed: {result}')
        return result['id']

class StripeSDK:
    # Simulated Stripe -- also cannot change this
    def create_payment_intent(self, amount: int, currency: str, payment_method: str) -> dict:
        return {'id': 'pi_stripe_xyz', 'status': 'succeeded'}

class StripeAdapter(PaymentGateway):
    def __init__(self) -> None: self._sdk = StripeSDK()
    def charge(self, amount: float, currency: str, token: str) -> str:
        result = self._sdk.create_payment_intent(
            amount=int(amount * 100), currency=currency.lower(), payment_method=token)
        if result['status'] != 'succeeded': raise RuntimeError(f'Stripe failed: {result}')
        return result['id']

def checkout(gateway: PaymentGateway, cart_total: float, token: str) -> str:
    txn_id = gateway.charge(cart_total, 'USD', token)
    print(f'  Charged ${cart_total} -> txn: {txn_id}')
    return txn_id

print('=== Braintree ===')
checkout(BraintreeAdapter(), 99.99, 'nonce_abc')
print('=== Stripe (migration: change 1 line, not 47) ===')
checkout(StripeAdapter(), 99.99, 'pm_stripe_xyz')

### Where This Is Seen in Real Frameworks

| Framework | Adapter usage |
|-----------|______________|
| **SQLAlchemy** | DBAPI adapters (`psycopg2`, `asyncpg`) -- all expose the same DBAPI2 interface |
| **Django auth** | `AUTHENTICATION_BACKENDS` -- custom backends adapt any auth to Django's `authenticate()` |
| **httpx** | `MockTransport` adapts test responses to real HTTP transport interface |
| **Celery** | Broker adapters (Redis, RabbitMQ, SQS) -- all adapt to the same `Transport` interface |
| **Logging** | `logging.Handler` subclasses -- each adapts a destination (file, Slack, Sentry) |

---
## 2 · Decorator

### Mental Model -- 'The Gift Wrapper'

```
WHAT   Attach new responsibilities to an object DYNAMICALLY,
       without changing the object's class.
WHY    Avoids deep inheritance for cross-cutting concerns.
HOW    Wrap the object, delegate to it, add behaviour before/after.
WHEN   Auth | rate limiting | caching | logging | retry | metrics
```

```
Request -> [auth]           -> checks JWT
        -> [rate_limit]     -> checks Redis counter
        -> [log]            -> logs request
        -> [actual handler] -> runs business logic

Each layer wraps the next -- onion-skin structure.
Fix auth once -> affects ALL 50 endpoints automatically.
```

In [ ]:
# BEFORE -- auth + rate-limit + log duplicated in every endpoint

_rate_counters: dict = {}

def get_product_BAD(product_id: int, token: str, user_id: str) -> dict:
    # Auth -- copy-pasted in 50 endpoints
    if token != 'valid_token': raise PermissionError('Unauthorized')
    # Rate limit -- copy-pasted in 50 endpoints
    key = (user_id, int(time.time() // 60))
    _rate_counters[key] = _rate_counters.get(key, 0) + 1
    if _rate_counters[key] > 100: raise RuntimeError('Rate limit exceeded')
    # Logging -- copy-pasted in 50 endpoints
    print(f'[LOG] get_product called by {user_id}')
    return {'id': product_id, 'name': 'Widget'}
# Fix auth? Edit 50 files. Miss one? Security hole.

In [ ]:
# AFTER -- Decorator Pattern (functional style, Python idiomatic)

def require_auth(fn: Callable) -> Callable:
    @functools.wraps(fn)
    def wrapper(*args, token: str = '', **kwargs):
        if token != 'valid_token': raise PermissionError('Unauthorized')
        return fn(*args, token=token, **kwargs)
    return wrapper

_counters: dict = {}

def rate_limit(max_per_minute: int = 100):
    def decorator(fn: Callable) -> Callable:
        @functools.wraps(fn)
        def wrapper(*args, user_id: str = 'anon', **kwargs):
            key = (user_id, int(time.time() // 60))
            _counters[key] = _counters.get(key, 0) + 1
            if _counters[key] > max_per_minute:
                raise RuntimeError(f'Rate limit exceeded for {user_id}')
            return fn(*args, user_id=user_id, **kwargs)
        return wrapper
    return decorator

def log_calls(fn: Callable) -> Callable:
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        print(f'[LOG] {fn.__name__} -- {(time.perf_counter()-t0)*1000:.1f}ms')
        return result
    return wrapper

@log_calls
@rate_limit(max_per_minute=100)
@require_auth
def get_product(product_id: int, *, token: str = '', user_id: str = '') -> dict:
    return {'id': product_id, 'name': 'Widget'}   # pure business logic

result = get_product(42, token='valid_token', user_id='user_1')
print('Result:', result)

### Where This Is Seen in Real Frameworks

| Framework | Decorator usage |
|-----------|----------------|
| **FastAPI** | `@app.get('/')` wraps your function; `Depends()` adds injectable decorators |
| **Django** | `@login_required`, `@permission_required`, `@cache_page` |
| **Celery** | `@app.task(bind=True, max_retries=3)` -- decorates with retry logic |
| **SQLAlchemy** | `@event.listens_for(Session, 'before_commit')` -- decorator adds hooks |
| **Python stdlib** | `@functools.lru_cache`, `@functools.cached_property`, `@property` |

---
## 3 · Facade

### Mental Model -- 'The Hotel Concierge'

```
WHAT   A single simplified interface to a complex subsystem.
WHY    Clients shouldn't need to know that placing an order involves
       Inventory + Payment + Warehouse + Email + Analytics.
HOW    Facade class calls subsystems in the right order.
WHEN   Application service layer | SDK simplification | API gateways
```

```
Client
  |
  v
OrderFacade.place_order(cart, token)
  +-> InventoryService.reserve(items)
  +-> PaymentService.charge(total, token)
  +-> WarehouseService.pick_and_pack(order_id)
  +-> EmailService.send_confirmation(email)
  +-> AnalyticsService.track('order_placed', order_id)
```

In [ ]:
# BEFORE -- endpoint directly orchestrates 5 subsystems

def checkout_endpoint_BAD(cart: dict, token: str, user_email: str) -> dict:
    if not all(item['qty'] <= 10 for item in cart['items']):
        raise RuntimeError('Out of stock')
    payment_id = f'txn_{int(time.time())}'
    # BUG: analytics fires BEFORE payment success check!
    print(f'[Analytics] order_started -- {user_email}')  # fires even if payment fails
    print(f'[Email] sending to {user_email}')
    order_id = f'ord_{int(time.time())}'
    print(f'[Warehouse] packing {order_id}')
    return {'order_id': order_id, 'payment': payment_id}
# 150-line endpoint, duplicated in every checkout variant.

In [ ]:
# AFTER -- Facade Pattern

class InventoryService:
    def reserve(self, items: list[dict]) -> bool:
        print(f'  [Inventory] reserved {len(items)} item(s)'); return True
    def release(self, items: list[dict]) -> None:
        print(f'  [Inventory] released {len(items)} item(s)')

class PaymentService:
    def charge(self, total: float, token: str) -> str:
        print(f'  [Payment] charged ${total:.2f}')
        return f'txn_{int(time.time())}'

class WarehouseService:
    def pick_and_pack(self, order_id: str, items: list[dict]) -> None:
        print(f'  [Warehouse] packing {order_id} ({len(items)} items)')

class EmailService:
    def send_confirmation(self, email: str, order_id: str) -> None:
        print(f'  [Email] -> {email}: order {order_id} confirmed')

class AnalyticsService:
    def track(self, event: str, order_id: str) -> None:
        print(f'  [Analytics] {event} -- {order_id}')


class OrderFacade:
    def __init__(self) -> None:
        self._inv, self._payment = InventoryService(), PaymentService()
        self._warehouse, self._email = WarehouseService(), EmailService()
        self._analytics = AnalyticsService()

    def place_order(self, cart: dict, token: str) -> dict:
        items = cart['items']
        if not self._inv.reserve(items):
            raise RuntimeError('Out of stock')
        try:
            txn_id = self._payment.charge(cart['total'], token)
        except Exception:
            self._inv.release(items); raise
        order_id = f'ord_{int(time.time())}'
        self._warehouse.pick_and_pack(order_id, items)
        self._email.send_confirmation(cart['email'], order_id)
        self._analytics.track('order_placed', order_id)  # AFTER payment
        return {'order_id': order_id, 'txn_id': txn_id}


facade = OrderFacade()
result = facade.place_order(
    {'items': [{'sku': 'W1', 'qty': 2}], 'total': 49.98,
     'email': 'alice@shopflow.com'}, 'tok_abc')
print('Order placed:', result)

### Where This Is Seen in Real Frameworks

| Framework | Facade usage |
|-----------|-------------|
| **Django** | Class-based views -- `View.dispatch()` coordinates request -> middleware -> response |
| **FastAPI** | Service layer in clean architecture -- `OrderService` hides repo + event bus + email |
| **SQLAlchemy** | `Session` -- one object hides connection pool + identity map + unit of work |
| **Celery** | `chain()` / `group()` -- facade over complex task graph |
| **AWS SDK** | `boto3.client('s3')` -- facade over HTTP signing + retries + pagination |

---
## 4 · Proxy

### Mental Model -- 'The Bank Card'

```
WHAT   An object that controls access to another object.
       The client talks to the proxy; the proxy decides whether/how to
       forward the call to the real object.
WHY    Transparent caching, lazy loading, access control --
       without changing the client or the real service.
HOW    Proxy implements the same interface as the real object.
WHEN   Caching proxies | virtual (lazy) proxies | protection proxies |
       remote proxies (hide network call behind a local object)
```

```
Client -> ProductProxy.get(id=1)
              |
              +-> cache hit?  YES -> return cached  (DB never called)
              +-> cache miss -> DB.get(id=1) -> store in cache -> return
```

**Proxy vs Decorator:** Both wrap an object with the same interface.
Proxy controls ACCESS (caching, auth, lazy load).
Decorator adds BEHAVIOUR (logging, metrics, retries).

In [ ]:
# BEFORE -- cache logic mixed into the service

class ProductService_BAD:
    _cache: dict = {}
    _ttl:   dict = {}

    def get_product(self, product_id: int) -> dict:
        now = time.monotonic()
        if product_id in self._cache and now - self._ttl[product_id] < 60:
            return self._cache[product_id]
        product = {'id': product_id, 'name': 'Widget', 'price': 9.99}
        self._cache[product_id] = product
        self._ttl[product_id]   = now
        return product
# ProductService now has two jobs: fetch data AND manage a cache.

In [ ]:
# AFTER -- Proxy Pattern

class ProductRepository(ABC):
    @abstractmethod
    def get_product(self, product_id: int) -> dict: ...
    @abstractmethod
    def save_product(self, product: dict) -> None: ...


class PostgresProductRepo(ProductRepository):
    def get_product(self, product_id: int) -> dict:
        print(f'  [DB] fetching product {product_id}')
        return {'id': product_id, 'name': 'Widget', 'price': 9.99, '_from': 'db'}
    def save_product(self, product: dict) -> None:
        print(f'  [DB] saved product {product["id"]}')


class CachingProductProxy(ProductRepository):
    def __init__(self, real: ProductRepository, ttl: float = 60) -> None:
        self._real = real
        self._ttl  = ttl
        self._cache: dict[int, dict] = {}
        self._times: dict[int, float] = {}

    def get_product(self, product_id: int) -> dict:
        now = time.monotonic()
        if product_id in self._cache and now - self._times[product_id] < self._ttl:
            print(f'  [Cache] HIT for product {product_id}')
            return {**self._cache[product_id], '_from': 'cache'}
        result = self._real.get_product(product_id)
        self._cache[product_id] = result
        self._times[product_id] = now
        return result

    def save_product(self, product: dict) -> None:
        self._cache.pop(product['id'], None)   # invalidate on write
        self._real.save_product(product)


repo: ProductRepository = CachingProductProxy(PostgresProductRepo(), ttl=60)

print('First call (miss):')
print(' ', repo.get_product(1)['_from'])
print('Second call (hit):')
print(' ', repo.get_product(1)['_from'])
print('After save (invalidated):')
repo.save_product({'id': 1, 'name': 'Widget v2', 'price': 12.99})
print(' ', repo.get_product(1)['_from'])  # miss again

### Where This Is Seen in Real Frameworks

| Framework | Proxy usage |
|-----------|------------|
| **SQLAlchemy** | Lazy-loaded relationships -- `order.user` is a proxy; DB hit on first access |
| **Django** | `SimpleLazyObject` -- wraps `request.user`; DB hit only when attribute accessed |
| **Redis / django-cache** | Cache backend is a proxy over the real service |
| **unittest.mock** | `MagicMock` -- a proxy that records all calls |
| **gRPC stubs** | Client stub is a proxy that turns local calls into remote calls |

---
## 5 · Composite

### Mental Model -- 'The File System'

```
WHAT   Treat individual objects and collections uniformly
       through a single interface.
WHY    Clients shouldn't need to know if they're dealing with a
       single item or a group of items.
HOW    Both leaf (single) and composite (group) implement the same interface.
       Composite delegates to its children.
WHEN   File systems | UI component trees | order discount rules |
       report sections | org charts
```

```
DiscountRule (interface: apply(total) -> float)
   |
   +-- PercentDiscount(10%)       <- Leaf
   +-- FlatDiscount($5)           <- Leaf
   +-- CompositeDiscount          <- Composite
       +-- BlackFridayDiscount
       +-- LoyaltyDiscount
```

In [ ]:
class DiscountRule(ABC):
    @abstractmethod
    def apply(self, total: float) -> float:
        # Return the discount amount (not the discounted total)
        ...
    def __repr__(self) -> str: return self.__class__.__name__


class PercentDiscount(DiscountRule):
    def __init__(self, pct: float) -> None: self.pct = pct
    def apply(self, total: float) -> float: return total * self.pct / 100
    def __repr__(self) -> str: return f'PercentDiscount({self.pct}%)'


class FlatDiscount(DiscountRule):
    def __init__(self, amount: float) -> None: self.amount = amount
    def apply(self, total: float) -> float: return min(self.amount, total)
    def __repr__(self) -> str: return f'FlatDiscount(${self.amount})'


class CompositeDiscount(DiscountRule):
    # Sums all child discounts -- client treats it like a single rule
    def __init__(self, name: str, *rules: DiscountRule) -> None:
        self.name, self.rules = name, list(rules)
    def add(self, rule: DiscountRule) -> 'CompositeDiscount':
        self.rules.append(rule); return self
    def apply(self, total: float) -> float:
        return sum(r.apply(total) for r in self.rules)
    def __repr__(self) -> str:
        return f'CompositeDiscount({self.name}: {self.rules})'


black_friday = CompositeDiscount('BlackFriday', PercentDiscount(20), FlatDiscount(15))
vip_deal     = CompositeDiscount('VIP', black_friday, PercentDiscount(5))

total = 200.0
for rule in [PercentDiscount(10), FlatDiscount(5), black_friday, vip_deal]:
    discount = rule.apply(total)
    print(f'{rule!r:45s} -> saves ${discount:.2f} on ${total:.2f}')

### Where This Is Seen in Real Frameworks

| Framework | Composite usage |
|-----------|----------------|
| **Django templates** | Template tag trees -- nodes render their children recursively |
| **HTML / DOM** | Every element is a composite -- `div.innerHTML` renders the whole subtree |
| **pytest** | `Suite` of `TestCase` -- `suite.run()` runs all children |
| **asyncio** | `asyncio.gather()` -- treat a group of coroutines as a single awaitable |
| **Pydantic validators** | `model_validator` chains -- each validator calls the next |